In [ ]:
!pip uninstall -y mcp mcp-types
!pip cache purge


Found existing installation: mcp 1.30.0
Uninstalling mcp-1.30.0:
  Successfully uninstalled mcp-1.30.0
Found existing installation: mcp-types 2.2.0
Uninstalling mcp-types-2.2.0:
  Successfully uninstalled mcp-types-2.2.0
Files removed: 70


In [ ]:
get_ipython().system('pip install mcp')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
import importlib.metadata
print(importlib.metadata.version("mcp"))

2.2.0


Agentic AI Project


In [ ]:
!pip uninstall -y mcp fastmcp langchain-mcp-adapters langgraph langchain-google-genai
!pip install -q "mcp>=1.0.0,<2.0.0" mcp-types langgraph langchain-google-genai nest_asyncio

Found existing installation: mcp 2.2.0
Uninstalling mcp-2.2.0:
  Successfully uninstalled mcp-2.2.0
Found existing installation: langgraph 1.2.11
Uninstalling langgraph-1.2.11:
  Successfully uninstalled langgraph-1.2.11
Found existing installation: langchain-google-genai 4.4.0
Uninstalling langchain-google-genai-4.4.0:
  Successfully uninstalled langchain-google-genai-4.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.7 MB/s eta 0:00:00


In [ ]:
import os
import csv
import asyncio
import nest_asyncio
from google.colab import userdata

# LangChain & LangGraph Imports
from langchain_core.tools import StructuredTool
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

nest_asyncio.apply()

# 1. Setup Gemini API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini_API_Key")

# 2. Initialize Gemini LLM (Use valid model name)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# -------------------------------------------------------------
# 3. Local Real-World Tools (MCP Primitive Logic)
# -------------------------------------------------------------
CSV_FILE = "expenses.csv"

def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Item", "Amount", "Category"])

def add_expense(item: str, amount: float, category: str) -> str:
    """Logs a new expense with item, amount, and category into CSV."""
    _initialize_csv()
    with open(CSV_FILE, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([item, amount, category])
    return f"Successfully logged expense: {item} - ${amount} ({category})"

def get_expenses() -> str:
    """Retrieves all logged expenses from the CSV file."""
    _initialize_csv()
    with open(CSV_FILE, mode='r') as f:
        reader = csv.reader(f)
        rows = list(reader)
    if len(rows) <= 1:
        return "No expenses recorded yet."
    return "\n".join([", ".join(row) for row in rows])

# -------------------------------------------------------------
# 4. Wrap Tools as LangChain Structured Tools for LangGraph
# -------------------------------------------------------------
mcp_tools = [
    StructuredTool.from_function(
        func=add_expense,
        name="add_expense",
        description="Logs a new expense with item, amount, and category."
    ),
    StructuredTool.from_function(
        func=get_expenses,
        name="get_expenses",
        description="Retrieves all logged expenses from the expense tracker."
    )
]

# -------------------------------------------------------------
# 5. Build & Execute LangGraph Agent
# -------------------------------------------------------------
agent = create_react_agent(llm, mcp_tools)

async def run_agentic_workflow():
    print("--- Task 1: Log an expense ---")
    prompt_1 = "I bought a pizza for $12.50. Category is Food."
    response_1 = await agent.ainvoke({"messages": [("user", prompt_1)]})

    # Correct attribute inspection for LangChain BaseMessage objects
    for msg in response_1["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")

    print("\n--- Task 2: Retrieve records ---")
    prompt_2 = "Show me all expenses logged so far."
    response_2 = await agent.ainvoke({"messages": [("user", prompt_2)]})

    for msg in response_2["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")

# Run Execution Loop
asyncio.run(run_agentic_workflow())

/tmp/ipykernel_4290/381508619.py:68: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, mcp_tools)
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- Task 1: Log an expense ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': "I've logged your expense: **pizza** for **$12.50** under **Food**.", 'extras': {'signature': 'EqwBCqkBARFNMg/yZNHeYosEybEyKnWfee0ljUvJYBALxBFMuT/AcBfXDCQhheL982vR6q601pbuPq7sr7dsgbsbmPqBX+9Dq4jTAyof9WJtYncfSWc16zwiPmChPMmqJk/4e2e3yRoC1vpJKVD+1HadqRVzwk/029XPjGRP2XMMfZssGTYUrQSjsYzjD7HSmOGRhtouhWYoTLCGrsAmGSivO9cLgXQcjeW7UDzeFg=='}}]

--- Task 2: Retrieve records ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': 'Here are all the expenses logged so far:\n\n* **Item:** Pizza\n* **Amount:** $12.50\n* **Category:** Food', 'extras': {'signature': 'EtQBCtEBARFNMg9Ah5bq3rfOHxrocNJxJ520KxG17jmw+9zs0xvwfN2At1G9/vVOLphp3sQNPG8wkV8jSsN1rqGJqxHTwrMAiKKH6kM2ciGdsXStOOInmJox0LjTtV0LarZg2xq6f2iJuNp/aGKrZDm+Ia7z7dO77408pChq8fK0r2OHWdq7RnBK63l2R/LZ1JSKE+Km5ymBOalD9FpODUp4tdJ3NO6UGirCwoEydO0mfq2NgJOYnd4QxVt9i6Vszl0jpNMELS3Dlz2qGqhq+B0D/m/de98='}}]
